# 16 — Hypothetical ball–defender closest-approach scenarios

**Question:** Under explicitly assumed constant ball speed and constant extrapolation of the defender's *recent* velocity, how close would the defender's projected position come to a straight hypothetical carrier-to-teammate trajectory?

**Not** an interception prediction, feasible-pass label, observed pass simulation, probability estimate, or estimate of defender capabilities. The candidate teammate position is fixed at possession start; it is **not** a model of receiver arrival. No future tracking, ball-pass outcomes, or downstream event labels are used.

Input: Notebook 05 canonical snapshots (`P1_canonical_decisions_match_*_ROLES.jsonl`), Notebook 11 integrated geometry (`P2_integrated_candidate_landscape_match_*.jsonl`), Notebook 15 defender vectors (`P3_motion_vectors_match_*.jsonl`), and their gates.

Output: `outputs/P3_kinematic_scenarios_match_*.jsonl` locally; upload `outputs/P3_kinematic_scenarios_gate.json`.

**Assumed prospective ball speeds:** 12, 20 and 28 metres/second. These are exploratory **scenario inputs**, not estimated from historical ball displacement and not asserted to represent the actual pass-speed distribution. All three must be shown as sensitivity analysis; none is a primary calibrated model.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime,timezone
import json,math,statistics
ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None:raise FileNotFoundError('Launch this notebook from within the SkillCorner project')
OUT=ROOT/'outputs'
SPEEDS=(12.0,20.0,28.0)
source=json.loads((OUT/'P3_motion_vectors_gate.json').read_text())
integrated=json.loads((OUT/'P2_integrated_candidate_gate.json').read_text())
MIDS=sorted(str(x['match_id']) for x in source['per_match'])
assert len(MIDS)==20 and source['totals']['decisions']==18792
assert integrated['totals']['pairs']==187570
print('Validated source gates:',len(MIDS),'matches')

Validated source gates: 20 matches


## Mathematical definition

For a candidate, take carrier position **A**, teammate's observed position **B**, and defender's observed position **D₀** at the same frame. Under scenario speed *u*, the hypothetical ball reaches fixed endpoint B after **T=‖B−A‖/u**. Its assumed position at time t is **A + u·e·t**, where **e=(B−A)/‖B−A‖**.

For defender observed retrospective velocity **v** over the preceding half-second, projected position is **D₀+v·t**. The closest separation is the analytic minimum of **‖(A−D₀)+(u·e−v)t‖** for **0 ≤ t ≤ T**. We compare against a stationary-defender projection v=0.

This model has no reaction, turning, acceleration, ball flight height, occlusion, goalkeeper rules, receiver motion or pitch constraints. A small separation is not an interception, and a large separation does not prove a pass is possible. Missing velocities remain missing for the motion scenario; they are **never replaced by zero**. A stationary defender is a separate named scenario.

In [2]:
def k(r):
    return (int(r['period']),int(r['frame']),str(r['possession_event_id']),str(r['carrier_player_id']))
def finite_xy(p):
    try:
        x,y=map(float,p)
        return (x,y) if math.isfinite(x) and math.isfinite(y) else None
    except (TypeError,ValueError):return None

def closest(A,B,D,v,ball_speed):
    length=math.hypot(B[0]-A[0],B[1]-A[1])
    if length<=0 or not math.isfinite(length):return None
    horizon=length/ball_speed
    ball_v=(ball_speed*(B[0]-A[0])/length,ball_speed*(B[1]-A[1])/length)
    rel0=(A[0]-D[0],A[1]-D[1])
    relv=(ball_v[0]-v[0],ball_v[1]-v[1])
    norm2=relv[0]**2+relv[1]**2
    t=0 if norm2==0 else min(horizon,max(0.0,-(rel0[0]*relv[0]+rel0[1]*relv[1])/norm2))
    return math.hypot(rel0[0]+relv[0]*t,rel0[1]+relv[1]*t)

def read_index(path):
    d={}
    with path.open(encoding='utf-8') as stream:
        for line_no,line in enumerate(stream,1):
            if not line.strip():continue
            try:r=json.loads(line)
            except json.JSONDecodeError as e:raise ValueError(f'{path}:{line_no}: {e}') from e
            key=k(r)
            if key in d:raise AssertionError(f'{path.name}: duplicate decision key {key}')
            d[key]=r
    return d

## Run all matches: exact decision join and scenario sensitivity

Per candidate, the output retains nearest stationary-defender separation, nearest constant-velocity separation (when measurable), and a both-endpoints-detected-velocity subset. **No threshold or binary pass-availability variable is assigned.**

Current defender coordinates are taken from canonical snapshots; retrospective velocity comes only from Notebook 15. A candidate is reconstructed exactly from the verified signed X and lateral Y endpoint displacements. Positional quality flags from Notebook 11 are retained.

In [3]:
def process(mid):
    cpath=OUT/f'P1_canonical_decisions_match_{mid}_ROLES.jsonl'
    vpath=OUT/f'P3_motion_vectors_match_{mid}.jsonl'
    ipath=OUT/f'P2_integrated_candidate_landscape_match_{mid}.jsonl'
    for p in (cpath,vpath,ipath):
        if not p.exists():raise FileNotFoundError(p)
    canonical=read_index(cpath); motion=read_index(vpath)
    if canonical.keys()!=motion.keys():
        raise AssertionError(f'{mid}: canonical and motion decision keys disagree')
    counts=Counter(); metrics={str(u):[] for u in SPEEDS}; seen=set()
    output=OUT/f'P3_kinematic_scenarios_match_{mid}.jsonl'
    with ipath.open(encoding='utf-8') as src,output.open('w',encoding='utf-8') as dest:
        for line_number,line in enumerate(src,1):
            r=json.loads(line); key=k(r)
            if key not in canonical:raise AssertionError(f'{mid}: missing canonical decision {key}')
            c=canonical[key]; m=motion[key]
            pair=(key,str(r['teammate_player_id']))
            if pair in seen:raise AssertionError(f'{mid}: duplicate candidate {pair}')
            seen.add(pair)
            A=finite_xy(c['carrier_xy'])
            assert A is not None,f'{mid} integrated row lacks valid carrier {key}'
            sign=int(r['source_attack_x_sign'])
            if sign not in (-1,1):raise AssertionError(f'{mid}: invalid orientation')
            B=(A[0]+float(r['signed_endpoint_progression_m'])/sign,
               A[1]+float(r['lateral_displacement_global_y_m']))
            length=math.hypot(B[0]-A[0],B[1]-A[1])
            if abs(length-float(r['pass_segment_length_m']))>1e-6:
                raise AssertionError(f'{mid}: reconstructed candidate length mismatch {pair}')
            coords={}
            for quality in ('detected','extrapolated'):
                for d in c.get(f'opponents_{quality}',[]):
                    pid=str(d['player_id']);D=finite_xy(d.get('xy'))
                    if pid in coords:raise AssertionError(f'{mid}: duplicated defender {pid}')
                    if D is not None:coords[pid]=D
            moving={str(d['player_id']):d for d in m['defenders']}
            if set(coords)-set(moving):raise AssertionError(f'{mid}: motion file lacks defenders')
            rec={'match_id':mid,'period':key[0],'frame':key[1],
              'possession_event_id':key[2],'carrier_player_id':key[3],
              'teammate_player_id':pair[1],
              'pass_segment_length_m':length,
              'signed_endpoint_progression_m':float(r['signed_endpoint_progression_m']),
              'carrier_detected':r['carrier_detected'],
              'teammate_quality':r['teammate_quality'],
              'geometry_status':r['geometry_status'],
              'scenario_separation_m':{}}
            for u in SPEEDS:
                stationary=[];dynamic=[];quality=[]
                for pid,D in coords.items():
                    static=closest(A,B,D,(0.0,0.0),u)
                    if static is not None:stationary.append(static)
                    vinfo=moving[pid]
                    v=finite_xy(vinfo.get('velocity_0_to_minus5_mps'))
                    if v is None:continue
                    z=closest(A,B,D,v,u)
                    if z is not None:
                        dynamic.append(z)
                        if vinfo.get('halfsec_both_detected') is True:quality.append(z)
                values={'stationary_defenders_min_m':min(stationary) if stationary else None,
                    'motion_observed_subset_min_m':min(dynamic) if dynamic else None,
                    'motion_detected_endpoint_subset_min_m':min(quality) if quality else None,
                    'n_stationary_defenders':len(stationary),'n_motion_defenders':len(dynamic),
                    'n_motion_detected_endpoint_defenders':len(quality)}
                rec['scenario_separation_m'][str(u)]=values
                counts[f'speed_{int(u)}_motion_available_pairs']+=bool(dynamic)
                counts[f'speed_{int(u)}_detected_motion_available_pairs']+=bool(quality)
                if dynamic:metrics[str(u)].append(min(dynamic))
            dest.write(json.dumps(rec,separators=(',',':'))+'\n')
            counts['candidate_pairs']+=1
    counts['eligible_decisions']=len({x[0] for x in seen})
    expected=next(d for d in integrated['per_match'] if str(d['match_id'])==mid)
    if counts['candidate_pairs']!=expected['pairs'] or counts['eligible_decisions']!=expected['eligible_decisions']:
        raise AssertionError(f'{mid}: decision/candidate count differs from Notebook 11')
    return {'match_id':mid,'counts':dict(counts),'scenario_descriptives':{
        s:{'n':len(vals),'median_m':statistics.median(vals) if vals else None}
        for s,vals in metrics.items()}}
results=[]
for mid in MIDS:
    report=process(mid);results.append(report)
    print(mid,report['counts']['candidate_pairs'],'candidate scenarios completed')

1874553 9560 candidate scenarios completed
1886347 9990 candidate scenarios completed
1899585 9290 candidate scenarios completed
1925299 10620 candidate scenarios completed
1927964 11390 candidate scenarios completed
1953632 9700 candidate scenarios completed
1959846 9450 candidate scenarios completed
1986691 9150 candidate scenarios completed
1996435 9970 candidate scenarios completed
1996436 7850 candidate scenarios completed
2006229 9770 candidate scenarios completed
2006363 9760 candidate scenarios completed
2007448 8620 candidate scenarios completed
2007721 9590 candidate scenarios completed
2010085 9350 candidate scenarios completed
2011166 7550 candidate scenarios completed
2013725 9170 candidate scenarios completed
2015213 8810 candidate scenarios completed
2016236 9590 candidate scenarios completed
2017461 8390 candidate scenarios completed


## Quality gate and interpretation

Comparison of stationary and constant-velocity scenarios illustrates sensitivity to **motion extrapolation**. It cannot validate either scenario or estimate interception ability. Candidate rows are clustered within decisions and matches; any future predictive evaluation must use possession/episode and match grouping.

In [4]:
totals=Counter()
for r in results:totals.update(r['counts'])
assert totals['candidate_pairs']==187570
assert totals['eligible_decisions']==18757
assert len(results)==20
report={'created_utc':datetime.now(timezone.utc).isoformat(),
 'purpose':'Hypothetical closest-separation kinematics under explicit ball speed and constant defender velocity assumptions',
 'n_matches':20,'ball_speed_scenarios_mps':list(SPEEDS),
 'totals':dict(totals),'per_match':results,
 'method':'Analytic minimum Euclidean distance during assumed straight-ball-flight horizon',
 'interpretation_constraints':[
    'Assumed ball speeds are sensitivity inputs, not calibrated pass speeds',
    'Retrospective defender velocity is extrapolated at constant value; no acceleration, anticipation or turning',
    'Stationary-defender baseline is distinct from missing-motion cases',
    'Fixed target is teammate location at decision time, not receiver position at ball arrival',
    'Small projected closest separation does not prove interception; large separation does not certify feasible pass',
    'Detected-endpoint restriction applies to motion, not necessarily all defender coordinates nor all intermediate frames',
    'The model does not account for defender reachable sets, ball flight height or trajectory variation',
    'No cutoffs, binary pass-opportunity labels, statistical tests, or pressing-coordination conclusions',
    'Candidate pairs share decisions and matches; no independent-sample inference']}
path=OUT/'P3_kinematic_scenarios_gate.json'
path.write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps({'matches':20,'candidate_pairs':totals['candidate_pairs'],
                 'eligible_decisions':totals['eligible_decisions'],
                 'speed_20_motion_available':totals['speed_20_motion_available_pairs']},indent=2))
print('Upload:',path)

{
  "matches": 20,
  "candidate_pairs": 187570,
  "eligible_decisions": 18757,
  "speed_20_motion_available": 185560
}
Upload: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P3_kinematic_scenarios_gate.json
